# 02 - EDA, KPIs & Funnel

Replaces the old v2 notebook. The problem with v2 wasn't the code, it was the questions: it measured "how are events distributed" when the real question is "what turns an impression into a click, and a click into a purchase." This version is built around that question directly, and folds EDA, KPI, and funnel into one notebook instead of three, since splitting them created an artificial wall - not going to build charts just to have something to hand off to a notebook 03.

Two corrections made while building this, both checked against the actual data first:
- The old "purchase share" chart computed `Purchases / All Events`, not a real conversion rate. Replaced everywhere with `Purchases / Impressions` and `Purchases / Clicks`. For the record: in this dataset the old and correct numbers happen to rank campaigns almost identically (correlation 0.9998), because impression share is itself flat across campaigns - so the old chart wasn't misleading in this specific case, but the formula was wrong and wouldn't hold on a less uniform dataset, so it's fixed regardless.
- "Top campaigns by event volume" is dropped as a finding - it correlates 0.9998 with number of ads in the campaign, so it's really just ranking by ad count. The replacement below checks whether campaign size predicts *conversion rate* (it doesn't, correlation -0.02) before drawing any conclusion from campaign size.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

events = pd.read_csv("events_clean.csv")
ads = pd.read_csv("ads_clean.csv")
campaigns = pd.read_csv("campaigns_clean.csv")

events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")
bad_timestamps = events["timestamp"].isna().sum()
print("Rows with unparseable timestamp:", bad_timestamps)
if bad_timestamps:
    events = events.dropna(subset=["timestamp"]).copy()

analysis = (
    events
    .merge(
        ads[["ad_id", "campaign_id", "ad_platform", "ad_type"]],
        on="ad_id", how="left", validate="many_to_one"
    )
    .merge(
        campaigns[["campaign_id", "name"]],
        on="campaign_id", how="left", validate="many_to_one"
    )
)

print("Analysis dataset:", analysis.shape)


# Part A - EDA

## A1. Data structure

In [ ]:
print("Events:", len(events))
print("Ads:", ads["ad_id"].nunique())
print("Campaigns:", campaigns["campaign_id"].nunique())
print("Campaigns with at least one ad:", ads["campaign_id"].nunique())
print("Platforms:", sorted(ads["ad_platform"].unique()))
print("Ad types:", sorted(ads["ad_type"].unique()))
print("Event types:", sorted(events["event_type"].unique()))


## A2. Event volume by type

Not a performance chart - this is the class imbalance we're working with. Impressions dominate, purchases are rare, and every rate calculated later has to be read against that.


In [ ]:
event_counts = analysis["event_type"].value_counts().sort_values()

plt.figure(figsize=(8, 5))
sns.barplot(x=event_counts.values, y=event_counts.index, hue=event_counts.index,
            palette="viridis", legend=False)
plt.title("Event Volume by Type")
plt.xlabel("Number of Events")
plt.ylabel("")
plt.tight_layout()
plt.show()


## A3. Ad inventory: platform x ad type

In [ ]:
inventory = pd.crosstab(ads["ad_platform"], ads["ad_type"])

plt.figure(figsize=(7, 4))
sns.heatmap(inventory, annot=True, fmt="d", cmap="mako")
plt.title("Ad Inventory: Platform x Ad Type")
plt.tight_layout()
plt.show()


## A4. Events over time

Using event timestamps, not campaign dates, since we already know those don't line up (most events fall outside their campaign's stated window).


In [ ]:
daily_events = events.set_index("timestamp").resample("D").size()

plt.figure(figsize=(12, 5))
color = sns.color_palette("crest", 5)[3]
plt.plot(daily_events.index, daily_events.values, color=color, linewidth=1.5)
plt.fill_between(daily_events.index, daily_events.values, alpha=0.25, color=color)
plt.title("Daily Event Volume")
plt.xlabel("Date")
plt.ylabel("Events per Day")
plt.tight_layout()
plt.show()

print("Observation window:", events["timestamp"].min().date(), "to", events["timestamp"].max().date())
print("Daily volume range:", daily_events.min(), "-", daily_events.max())


## A5. How activity is spread across ads

Impressions per ad are tight and even - every ad gets roughly the same exposure. Purchases per ad are proportionally much more spread out, which is the first hint that conversion, not exposure, is where the real differences show up.


In [ ]:
ad_activity = analysis.pivot_table(
    index="ad_id", columns="event_type", values="event_id",
    aggfunc="count", fill_value=0
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(ad_activity["Impression"], bins=20, color=sns.color_palette("rocket")[2], ax=axes[0])
axes[0].set_title("Impressions per Ad")
axes[0].set_xlabel("Impressions")

sns.histplot(ad_activity["Purchase"], bins=15, color=sns.color_palette("rocket")[4], ax=axes[1])
axes[1].set_title("Purchases per Ad")
axes[1].set_xlabel("Purchases")

plt.tight_layout()
plt.show()

print("Impressions/ad - mean:", round(ad_activity["Impression"].mean(),1), "std:", round(ad_activity["Impression"].std(),1))
print("Purchases/ad - mean:", round(ad_activity["Purchase"].mean(),1), "std:", round(ad_activity["Purchase"].std(),1))


## A6. Campaign size vs. campaign performance

Left: raw impressions vs. purchases per campaign - strongly correlated, but that's mostly because bigger campaigns (more ads) generate more of everything. Right: purchase rate vs. number of ads - this is what actually tests whether bigger campaigns convert better. They don't; the correlation is close to zero. Campaign size drives volume, not quality.


In [ ]:
campaign_activity = analysis.pivot_table(
    index="campaign_id", columns="event_type", values="event_id",
    aggfunc="count", fill_value=0
)
campaign_activity["unique_ads"] = analysis.groupby("campaign_id")["ad_id"].nunique()
campaign_activity["PurchaseRate"] = campaign_activity["Purchase"] / campaign_activity["Impression"] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(
    data=campaign_activity, x="Impression", y="Purchase",
    hue="unique_ads", palette="viridis", s=80, ax=axes[0]
)
axes[0].set_title("Impressions vs. Purchases (by campaign)")

sns.scatterplot(
    data=campaign_activity, x="unique_ads", y="PurchaseRate",
    color=sns.color_palette("flare")[3], s=80, ax=axes[1]
)
axes[1].set_title("Purchase Rate vs. Number of Ads")
axes[1].set_xlabel("Ads in Campaign")
axes[1].set_ylabel("Purchase Rate (%)")

plt.tight_layout()
plt.show()

print("corr(impressions, purchases):", round(campaign_activity["Impression"].corr(campaign_activity["Purchase"]), 3))
print("corr(unique_ads, purchase rate):", round(campaign_activity["unique_ads"].corr(campaign_activity["PurchaseRate"]), 3))


### Note on the dimensions we're not charting

Platform, ad type, day of week, and time of day all show limited variation in raw event composition (each stays within about half a percentage point of the overall average). They're not dropped from the analysis below - they still get real KPI numbers - but they don't get four separate composition heatmaps here, since those would all say the same thing: everywhere looks roughly the same.


# Part B - KPI Analysis

## B1. Metric definitions

- **CTR** = Clicks / Impressions
- **Click-to-Purchase Rate** = Purchases / Clicks
- **Purchase Rate** = Purchases / Impressions


## B2. KPIs by platform

In [ ]:
platform_kpi = analysis.pivot_table(
    index="ad_platform", columns="event_type", values="event_id",
    aggfunc="count", fill_value=0
)
platform_kpi["CTR"] = platform_kpi["Click"] / platform_kpi["Impression"] * 100
platform_kpi["ClickToPurchase"] = platform_kpi["Purchase"] / platform_kpi["Click"] * 100
platform_kpi["PurchaseRate"] = platform_kpi["Purchase"] / platform_kpi["Impression"] * 100

display(platform_kpi[["Impression", "Click", "Purchase", "CTR", "ClickToPurchase", "PurchaseRate"]])

platform_long = (
    platform_kpi[["CTR", "ClickToPurchase", "PurchaseRate"]]
    .reset_index()
    .melt(id_vars="ad_platform", var_name="metric", value_name="value")
)

plt.figure(figsize=(9, 5))
sns.barplot(data=platform_long, x="metric", y="value", hue="ad_platform", palette="Set2")
plt.title("KPIs by Platform")
plt.ylabel("Rate (%)")
plt.xlabel("")
plt.tight_layout()
plt.show()


## B3. KPIs by ad type

In [ ]:
adtype_kpi = analysis.pivot_table(
    index="ad_type", columns="event_type", values="event_id",
    aggfunc="count", fill_value=0
)
adtype_kpi["CTR"] = adtype_kpi["Click"] / adtype_kpi["Impression"] * 100
adtype_kpi["ClickToPurchase"] = adtype_kpi["Purchase"] / adtype_kpi["Click"] * 100
adtype_kpi["PurchaseRate"] = adtype_kpi["Purchase"] / adtype_kpi["Impression"] * 100

display(adtype_kpi[["Impression", "Click", "Purchase", "CTR", "ClickToPurchase", "PurchaseRate"]])

adtype_long = (
    adtype_kpi[["CTR", "ClickToPurchase", "PurchaseRate"]]
    .reset_index()
    .melt(id_vars="ad_type", var_name="metric", value_name="value")
)

plt.figure(figsize=(10, 5))
sns.barplot(data=adtype_long, x="metric", y="value", hue="ad_type", palette="crest")
plt.title("KPIs by Ad Type")
plt.ylabel("Rate (%)")
plt.xlabel("")
plt.tight_layout()
plt.show()


## B4. KPIs by campaign

Campaign-level click-to-purchase rate shows the widest variation among the breakdowns examined so far (roughly 2.4% to 8.4%).


In [ ]:
campaign_kpi = (
    analysis
    .pivot_table(index="campaign_id", columns="event_type", values="event_id", aggfunc="count", fill_value=0)
    .join(campaigns.set_index("campaign_id")["name"])
)
campaign_kpi["CTR"] = campaign_kpi["Click"] / campaign_kpi["Impression"] * 100
campaign_kpi["ClickToPurchase"] = campaign_kpi["Purchase"] / campaign_kpi["Click"] * 100
campaign_kpi["PurchaseRate"] = campaign_kpi["Purchase"] / campaign_kpi["Impression"] * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics = ["CTR", "ClickToPurchase", "PurchaseRate"]
cmaps = ["viridis", "mako", "rocket"]

for ax, metric, cmap in zip(axes, metrics, cmaps):
    top = campaign_kpi.nlargest(5, metric)
    bottom = campaign_kpi.nsmallest(5, metric)
    combo = pd.concat([top, bottom]).sort_values(metric)
    colors = sns.color_palette(cmap, len(combo))
    ax.barh(combo["name"], combo[metric], color=colors)
    ax.set_title(f"{metric} - Highest & Lowest 5 Campaigns")
    ax.set_xlabel("Rate (%)")

plt.tight_layout()
plt.show()


# Part C - Funnel Analysis

## C1. Overall funnel

In [ ]:
overall = analysis["event_type"].value_counts()
impressions, clicks, purchases = overall["Impression"], overall["Click"], overall["Purchase"]

print("Impressions:", impressions)
print("Clicks:", clicks)
print("Purchases:", purchases)
print("CTR:", round(clicks / impressions * 100, 2), "%")
print("Click-to-Purchase Rate:", round(purchases / clicks * 100, 2), "%")
print("Purchase Rate:", round(purchases / impressions * 100, 2), "%")

funnel_labels = ["Impressions", "Clicks", "Purchases"]
funnel_values = [impressions, clicks, purchases]
colors = sns.color_palette("rocket", 3)

plt.figure(figsize=(8, 5))
plt.barh(funnel_labels[::-1], funnel_values[::-1], color=colors[::-1])
for i, v in enumerate(funnel_values[::-1]):
    plt.text(v, i, f"  {v:,}", va="center")
plt.title("Advertising Event Funnel: Impressions -> Clicks -> Purchases")
plt.xlabel("Count")
plt.tight_layout()
plt.show()


**Note:** this is an event-based funnel built from independent event counts, not a verified user-level journey. The dataset does not confirm that the same user who saw an impression is the one who clicked or purchased, so read this as funnel-shaped volume, not a traced path per user.


## C2. Funnel by platform

In [ ]:
platform_funnel = (
    analysis.pivot_table(index="ad_platform", columns="event_type", values="event_id", aggfunc="count", fill_value=0)
    [["Impression", "Click", "Purchase"]]
    .reset_index()
    .melt(id_vars="ad_platform", var_name="stage", value_name="count")
)

plt.figure(figsize=(9, 5))
sns.barplot(data=platform_funnel, x="stage", y="count", hue="ad_platform", palette="magma")
plt.yscale("log")
plt.title("Funnel by Platform (log scale)")
plt.ylabel("Count (log)")
plt.xlabel("")
plt.tight_layout()
plt.show()


## Summary

- Impressions account for about 85% of all events, while purchases account for about 0.5% of all events. Relative to impressions specifically, the purchase rate is about 0.60%.
- Platform and ad type show relatively small differences in CTR and purchase rate - Facebook and Instagram, and all four ad formats, land within a point of each other.
- Campaign is where real variation shows up, especially in click-to-purchase rate. Bigger campaigns (more ads) generate more volume but not a higher conversion rate - size and quality are unrelated here.
- Next: aggregate these tables (campaign-level and platform-level KPIs) for the Power BI dashboard.
